When I was building the SQL for `demo2`, I had a discrepancy between the SQL query with joining/grouping, and the basic date query with filtering done with dataframe. So I have left the work I did for that here

In [1]:
%useLatestDescriptors
%use dataframe, kandy

// %use dataframe(1.0.0-Beta4n), kandy(0.8.3)

In [2]:
import util.Helpers

val helpers = Helpers()

The key learning from this was that the where clause for the join needs the same filtering as the outer query

In [3]:
fun queryByTimePeriodAndEntries(startYear: String, endYear: String, liftersPerWeightclass: Int) =
    """
SELECT
    pd.*
FROM
    powerlifting_data pd
        JOIN
    (
        SELECT
            meetname,
            date,
            weightclasskg,
            division,
            COUNT(*) AS lifter_count
        FROM
            powerlifting_data
        WHERE
            event = 'SBD' AND place != 'NS'
            AND date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
            AND totalkg IS NOT NULL
        GROUP BY
            meetname, date, weightclasskg, division
        HAVING
            COUNT(*) >= $liftersPerWeightclass
    ) AS qualified_classes
    ON pd.meetname = qualified_classes.meetname
        AND pd.date = qualified_classes.date
        AND pd.weightclasskg = qualified_classes.weightclasskg
        AND pd.division = qualified_classes.division
WHERE
pd.event = 'SBD' AND pd.place != 'NS'
  AND pd.date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
  AND pd.totalkg IS NOT NULL;
    """

In [4]:
fun queryByTimePeriod(startYear: String, endYear: String) =
    """
SELECT
    pd.*
FROM
    powerlifting_data pd
WHERE
  pd.date BETWEEN '$startYear-01-01' AND '$endYear-12-31'
    """

By delcaring The `LifterData` interface with the `@DataSchema` annotation, we can make use of API extension properties so our results are fully typed and checked at compile time by the compiler plugin.

In [5]:
import java.util.Date
import org.jetbrains.kotlinx.dataframe.annotations.*

@DataSchema
interface LifterData {
    val place: String
    val meetname: String
    val date: Date
    val event: String
    val weightclasskg: String?
    val entries: Int
    val division: String?
    val squat1kg: Double?
    val squat2kg: Double?
    val squat3kg: Double?
    val bench1kg: Double?
    val bench2kg: Double?
    val bench3kg: Double?
    val deadlift1kg: Double?
    val deadlift2kg: Double?
    val deadlift3kg: Double?
    val best3squatkg: Double?
    val best3benchkg: Double?
    val best3deadliftkg: Double?
    val totalkg: Double?
    val successfulLifts: Int
}

Comparison time

In [7]:
val queryGrouped = queryByTimePeriodAndEntries(startYear = "2025", endYear = "2025", liftersPerWeightclass = 3)
val dataGrouped: DataFrame<*> = helpers.fetchResults(queryGrouped)

In [8]:
val queryUnGrouped = queryByTimePeriod(startYear = "2025", endYear = "2025")
val dataUnGrouped: DataFrame<*> = helpers.fetchResults(queryUnGrouped)

This is what the `join` statement in the SQL does. It groups the rows unique by meetname, weight class and division then counts how many lifters (enrties are in each weight class at that event

In [9]:
val entriesPerClass = dataUnGrouped.cast<LifterData>()
    .filter { event == "SBD" && place != "NS" && totalkg != null }
    .groupBy { meetname and date and weightclasskg and division } // using API extension properties is very nice
        .aggregate {
        count().into("entries")
    }

In [10]:
val unGroupedDataWithEntries = dataUnGrouped.cast<LifterData>().innerJoin(entriesPerClass)

unGroupedDataWithEntries.count()

72688

In [11]:
val unGroupedFiltered =
    unGroupedDataWithEntries.cast<LifterData>()
        .filter {
            event == "SBD" && place != "NS" && totalkg != null && it.entries >= 3
        }

unGroupedFiltered.count()

48623

In [12]:
dataGrouped.count()

47985

Turns out we still have a discrepancy between the SQL join and the dataframe one

In [13]:
dataGrouped.count() - unGroupedFiltered.count()

-638

Reading the Postgres docs, it will silently remove nulls from the join

In [14]:
dataGrouped.filter { weightclasskg == null }.count()

0

Where as the innerJoin of dataframe will not remove nulls

In [15]:
val nullWeightClasses = unGroupedFiltered.filter { weightclasskg == null }

nullWeightClasses.count()

638